# Notebook 01 — Preprocessing

## Projet : C-MARL pour la réduction des absences aux rendez-vous médicaux

Ce notebook prépare le dataset médical pour les trois stages du projet :

- Stage Easy
- Stage Medium
- Stage Hard

À la fin, trois fichiers seront sauvegardés :

- `easy_data.csv`
- `medium_data.csv`
- `hard_data.csv`

Dataset utilisé : **Medical Appointment No Shows** de Kaggle.


## Cellule 1 — Imports

On importe les bibliothèques nécessaires pour lire, nettoyer et transformer les données.

In [20]:
import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder

## Cellule 2 — Chemins du projet

Le notebook doit être placé dans le dossier `notebooks/`.

Structure attendue :

```text
medical_c_marl_project/
├── data/
│   └── KaggleV2-May-2016.csv
├── results/
└── notebooks/
    └── 01_preprocessing.ipynb
```

In [21]:
PROJECT_PATH = os.path.abspath("..")

DATA_PATH = os.path.join(PROJECT_PATH, "data", "KaggleV2-May-2016.csv")
RESULTS_DIR = os.path.join(PROJECT_PATH, "results")

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Project path:", PROJECT_PATH)
print("Dataset path:", DATA_PATH)
print("Results path:", RESULTS_DIR)

Project path: C:\Users\UltraPc\Documents\ML PROBA\C-marl
Dataset path: C:\Users\UltraPc\Documents\ML PROBA\C-marl\data\KaggleV2-May-2016.csv
Results path: C:\Users\UltraPc\Documents\ML PROBA\C-marl\results


## Cellule 3 — Vérifier que le fichier existe

Cette cellule vérifie que le dataset est bien placé dans le dossier `data/`.

In [22]:
if os.path.exists(DATA_PATH):
    print("Dataset trouvé.")
else:
    print("Dataset introuvable.")
    print("Vérifie que le fichier est bien dans :", DATA_PATH)

Dataset trouvé.


## Cellule 4 — Charger le dataset

On charge le fichier CSV avec Pandas.

In [23]:
df = pd.read_csv(DATA_PATH)

print("Shape initiale:", df.shape)
df.head()

Shape initiale: (110527, 14)


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


## Cellule 5 — Afficher les colonnes

Cette étape permet de vérifier les noms exacts des colonnes avant le nettoyage.

In [24]:
print("Colonnes du dataset:")
print(df.columns.tolist())

Colonnes du dataset:
['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay', 'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'No-show']


## Cellule 6 — Nettoyage des noms de colonnes

On corrige certains noms de colonnes pour rendre le code plus propre :

- `Hipertension` devient `Hypertension`
- `Handcap` devient `Handicap`
- `No-show` devient `No_show`

In [25]:
df = df.copy()

df = df.rename(columns={
    "Hipertension": "Hypertension",
    "Handcap": "Handicap",
    "No-show": "No_show"
})

print("Colonnes après correction:")
print(df.columns.tolist())

Colonnes après correction:
['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay', 'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hypertension', 'Diabetes', 'Alcoholism', 'Handicap', 'SMS_received', 'No_show']


## Cellule 7 — Conversion des dates

Les colonnes `ScheduledDay` et `AppointmentDay` sont transformées en format date.

In [26]:
df["ScheduledDay"] = pd.to_datetime(df["ScheduledDay"], errors="coerce")
df["AppointmentDay"] = pd.to_datetime(df["AppointmentDay"], errors="coerce")

df = df.dropna(subset=["ScheduledDay", "AppointmentDay"])

print("Types après conversion:")
print(df[["ScheduledDay", "AppointmentDay"]].dtypes)

Types après conversion:
ScheduledDay      datetime64[ns, UTC]
AppointmentDay    datetime64[ns, UTC]
dtype: object


## Cellule 8 — Création de la variable `WaitingDays`

`WaitingDays` représente le nombre de jours entre la date de planification et la date du rendez-vous.

Cette variable sera surtout utilisée dans le stage **Hard**.

In [27]:
df["WaitingDays"] = (
    df["AppointmentDay"].dt.normalize() - df["ScheduledDay"].dt.normalize()
).dt.days

df[["ScheduledDay", "AppointmentDay", "WaitingDays"]].head()

,ScheduledDay,AppointmentDay,WaitingDays
0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,0
1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,0
2,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,0
3,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,0
4,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,0


## Cellule 9 — Suppression des valeurs incorrectes

On supprime les lignes avec :

- âge négatif ;
- délai d’attente négatif.

In [28]:
print("Avant suppression:", df.shape)

df = df[df["Age"] >= 0]
df = df[df["WaitingDays"] >= 0]

print("Après suppression:", df.shape)

Avant suppression: (110527, 15)
Après suppression: (110521, 15)


## Cellule 10 — Transformation de la variable cible `No_show`

Dans le dataset :

- `No` signifie que le patient est venu ;
- `Yes` signifie que le patient n’est pas venu.

On transforme donc :

- `0` = patient présent ;
- `1` = patient absent.

In [29]:
df["No_show"] = df["No_show"].map({
    "No": 0,
    "Yes": 1
})

df["No_show"].value_counts()

No_show
0    88207
1    22314
Name: count, dtype: int64

## Cellule 11 — Création du jour de rendez-vous

`AppointmentWeekday` indique le jour de la semaine :

- 0 = lundi
- 1 = mardi
- 2 = mercredi
- 3 = jeudi
- 4 = vendredi
- 5 = samedi
- 6 = dimanche

In [30]:
df["AppointmentWeekday"] = df["AppointmentDay"].dt.dayofweek

df[["AppointmentDay", "AppointmentWeekday"]].head()

,AppointmentDay,AppointmentWeekday
0,2016-04-29 00:00:00+00:00,4
1,2016-04-29 00:00:00+00:00,4
2,2016-04-29 00:00:00+00:00,4
3,2016-04-29 00:00:00+00:00,4
4,2016-04-29 00:00:00+00:00,4


## Cellule 12 — Encodage des variables textuelles

Le Q-learning utilise des valeurs numériques.

On transforme donc :

- `Gender` en valeur numérique ;
- `Neighbourhood` en valeur numérique.

In [31]:
label_encoder_gender = LabelEncoder()
label_encoder_neighbourhood = LabelEncoder()

df["Gender"] = label_encoder_gender.fit_transform(df["Gender"])
df["Neighbourhood"] = label_encoder_neighbourhood.fit_transform(df["Neighbourhood"])

df[["Gender", "Neighbourhood"]].head()

,Gender,Neighbourhood
0,0,39
1,1,39
2,0,45
3,0,54
4,0,39


## Cellule 13 — Discrétisation de l’âge

On transforme l’âge en groupes simples :

- 0 = 0 à 18 ans
- 1 = 19 à 40 ans
- 2 = 41 à 60 ans
- 3 = plus de 60 ans

In [32]:
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[-1, 18, 40, 60, 120],
    labels=[0, 1, 2, 3]
).astype(int)

df[["Age", "AgeGroup"]].head()

,Age,AgeGroup
0,62,3
1,56,2
2,62,3
3,8,0
4,56,2


## Cellule 14 — Discrétisation du délai d’attente

On transforme `WaitingDays` en groupes :

- 0 = 0 à 7 jours
- 1 = 8 à 15 jours
- 2 = 16 à 30 jours
- 3 = plus de 30 jours

In [33]:
df["WaitingGroup"] = pd.cut(
    df["WaitingDays"],
    bins=[-1, 7, 15, 30, 365],
    labels=[0, 1, 2, 3]
).astype(int)

df[["WaitingDays", "WaitingGroup"]].head()

,WaitingDays,WaitingGroup
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0


## Cellule 15 — Création de `MedicalRisk`

`MedicalRisk` indique si le patient possède au moins une condition médicale dans le dataset.

Ce n’est pas un diagnostic médical. C’est seulement une variable simplifiée pour la simulation.

In [34]:
df["MedicalRisk"] = (
    df["Hypertension"] +
    df["Diabetes"] +
    df["Alcoholism"] +
    df["Handicap"]
)

df["MedicalRisk"] = df["MedicalRisk"].apply(lambda x: 1 if x > 0 else 0)

df[["Hypertension", "Diabetes", "Alcoholism", "Handicap", "MedicalRisk"]].head()

,Hypertension,Diabetes,Alcoholism,Handicap,MedicalRisk
0,1,0,0,0,1
1,0,0,0,0,0
2,0,0,0,0,0
3,0,0,0,0,0
4,1,1,0,0,1


## Cellule 16 — Vérification finale des données

On vérifie la taille finale, les valeurs manquantes et la distribution de la variable cible.

In [35]:
print("Shape finale:", df.shape)

print("Valeurs manquantes:")
print(df.isnull().sum())

print("Distribution de No_show:")
print(df["No_show"].value_counts())

df.head()

Shape finale: (110521, 19)
Valeurs manquantes:
PatientId             0
AppointmentID         0
Gender                0
ScheduledDay          0
AppointmentDay        0
Age                   0
Neighbourhood         0
Scholarship           0
Hypertension          0
Diabetes              0
Alcoholism            0
Handicap              0
SMS_received          0
No_show               0
WaitingDays           0
AppointmentWeekday    0
AgeGroup              0
WaitingGroup          0
MedicalRisk           0
dtype: int64
Distribution de No_show:
No_show
0    88207
1    22314
Name: count, dtype: int64


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hypertension,Diabetes,Alcoholism,Handicap,SMS_received,No_show,WaitingDays,AppointmentWeekday,AgeGroup,WaitingGroup,MedicalRisk
0,2.987250e+13,5642903,0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62,39,0,1,0,0,0,0,0,0,4,3,0,1
1,5.589978e+14,5642503,1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56,39,0,0,0,0,0,0,0,0,4,2,0,0
2,4.262962e+12,5642549,0,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62,45,0,0,0,0,0,0,0,0,4,3,0,0
3,8.679512e+11,5642828,0,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8,54,0,0,0,0,0,0,0,0,4,0,0,0
4,8.841186e+12,5642494,0,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56,39,0,1,1,0,0,0,0,0,4,2,0,1


## Cellule 17 — Création du Stage Easy

Le niveau **Easy** utilise peu de variables :

- `AgeGroup`
- `Gender`
- `SMS_received`
- `No_show`

In [36]:
easy_data = df[
    [
        "AgeGroup",
        "Gender",
        "SMS_received",
        "No_show"
    ]
].copy()

print("Easy data shape:", easy_data.shape)
easy_data.head()

Easy data shape: (110521, 4)


,AgeGroup,Gender,SMS_received,No_show
0,3,0,0,0
1,2,1,0,0
2,3,0,0,0
3,0,0,0,0
4,2,0,0,0


## Cellule 18 — Création du Stage Medium

Le niveau **Medium** ajoute des variables médicales :

- `Hypertension`
- `Diabetes`
- `Alcoholism`
- `Handicap`
- `MedicalRisk`

In [37]:
medium_data = df[
    [
        "AgeGroup",
        "Gender",
        "SMS_received",
        "Scholarship",
        "Hypertension",
        "Diabetes",
        "Alcoholism",
        "Handicap",
        "MedicalRisk",
        "No_show"
    ]
].copy()

print("Medium data shape:", medium_data.shape)
medium_data.head()

Medium data shape: (110521, 10)


,AgeGroup,Gender,SMS_received,Scholarship,Hypertension,Diabetes,Alcoholism,Handicap,MedicalRisk,No_show
0,3,0,0,0,1,0,0,0,1,0
1,2,1,0,0,0,0,0,0,0,0
2,3,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0
4,2,0,0,0,1,1,0,0,1,0


## Cellule 19 — Création du Stage Hard

Le niveau **Hard** utilise le plus d’informations :

- variables personnelles ;
- variables médicales ;
- `WaitingGroup` ;
- `AppointmentWeekday` ;
- `Neighbourhood`.

In [38]:
hard_data = df[
    [
        "AgeGroup",
        "Gender",
        "SMS_received",
        "Scholarship",
        "Hypertension",
        "Diabetes",
        "Alcoholism",
        "Handicap",
        "MedicalRisk",
        "WaitingGroup",
        "AppointmentWeekday",
        "Neighbourhood",
        "No_show"
    ]
].copy()

print("Hard data shape:", hard_data.shape)
hard_data.head()

Hard data shape: (110521, 13)


,AgeGroup,Gender,SMS_received,Scholarship,Hypertension,Diabetes,Alcoholism,Handicap,MedicalRisk,WaitingGroup,AppointmentWeekday,Neighbourhood,No_show
0,3,0,0,0,1,0,0,0,1,0,4,39,0
1,2,1,0,0,0,0,0,0,0,0,4,39,0
2,3,0,0,0,0,0,0,0,0,0,4,45,0
3,0,0,0,0,0,0,0,0,0,0,4,54,0
4,2,0,0,0,1,1,0,0,1,0,4,39,0


## Cellule 20 — Comparaison des trois stages

Cette cellule montre que la complexité augmente progressivement de Easy vers Hard.

In [39]:
stage_summary = pd.DataFrame({
    "Stage": ["Easy", "Medium", "Hard"],
    "Nombre de lignes": [
        easy_data.shape[0],
        medium_data.shape[0],
        hard_data.shape[0]
    ],
    "Nombre de colonnes": [
        easy_data.shape[1],
        medium_data.shape[1],
        hard_data.shape[1]
    ]
})

stage_summary

,Stage,Nombre de lignes,Nombre de colonnes
0,Easy,110521,4
1,Medium,110521,10
2,Hard,110521,13


## Cellule 21 — Sauvegarder les trois datasets

On sauvegarde les datasets préparés pour les notebooks suivants :

- `02_environment.ipynb`
- `03_agents_training.ipynb`
- `04_evaluation_results.ipynb`

In [40]:
easy_path = os.path.join(RESULTS_DIR, "easy_data.csv")
medium_path = os.path.join(RESULTS_DIR, "medium_data.csv")
hard_path = os.path.join(RESULTS_DIR, "hard_data.csv")

easy_data.to_csv(easy_path, index=False)
medium_data.to_csv(medium_path, index=False)
hard_data.to_csv(hard_path, index=False)

print("Fichiers sauvegardés avec succès:")
print(easy_path)
print(medium_path)
print(hard_path)

Fichiers sauvegardés avec succès:
C:\Users\UltraPc\Documents\ML PROBA\C-marl\results\easy_data.csv
C:\Users\UltraPc\Documents\ML PROBA\C-marl\results\medium_data.csv
C:\Users\UltraPc\Documents\ML PROBA\C-marl\results\hard_data.csv


## Cellule 22 — Vérifier que les fichiers sont bien créés

In [41]:
print("Contenu du dossier results:")

for file in os.listdir(RESULTS_DIR):
    print(file)

Contenu du dossier results:
easy_data.csv
hard_data.csv
medium_data.csv


## Cellule 23 — Fin du notebook

In [42]:
print("Notebook 01 terminé avec succès.")
print("Les datasets Easy, Medium et Hard sont prêts pour le Notebook 02.")

Notebook 01 terminé avec succès.
Les datasets Easy, Medium et Hard sont prêts pour le Notebook 02.
